# Interfaces en Jupyter

---

<style>
      h1, h2, h3, h4, h5, h6,.imagen{
        text-align: center;
      }
 img{width: 50%; height: 50%;}
 body{
  transform: scale(1.3);
}
</style>

In [1]:
%pip install gradio

: 

In [ ]:
import ipywidgets as widgets
from IPython  import display



## Indice
- [Jupyter](#Formularios-Colab-y-Jupyter)
  - [Indice](#Indice)
    - [IPython Widgets](#IPython-Widgets)
    - [Gradio](#Gradio)
    - [Stable Diffusion](#Stable-Diffusion)
---


## IPython Widgets

Los widgets son objetos de Python que se utilizan para construir aplicaciones interactivas. Estos se emplean para crear interfaces gráficas de usuario (GUI) en los cuadernos de Jupyter. Los widgets pueden utilizarse para mostrar y manipular elementos como botones, casillas de selección, menús desplegables, barras de progreso, entre otros.

In [1]:
# Ejemplo, mostrar imagen

display.Image(url="https://www.python.org/static/community_logos/python-logo.png")


AttributeError: 'function' object has no attribute 'Image'

In [ ]:
#youtube

display.YouTubeVideo('gTNrBp_Ckio')

In [ ]:
#Audio

display.Audio(url="https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3", autoplay=False)



In [ ]:
#Codigo

display.Code(data="print('Hello World')", language="python")

In [ ]:
## Slider Int

w = widgets.IntSlider()
display.display(w)


# leer el valor del slider

w.value


In [ ]:
## Slider Float

w = widgets.FloatSlider(
      value=7.5,
    min=5.0,
    max=10.0,
    step=0.1
)
display.display(w)



In [ ]:
## Texto

w = widgets.Text( placeholder='placeholder',disabled=False)
display.display(w)

In [ ]:
## HTML

w = widgets.HTML(value="Hello <b>World</b>")
display.display(w)

In [ ]:
html_code = """
<!DOCTYPE html>
<html>
<head>
    <title>Mi Página</title>
    <style>
        .htmlcode{background-color: powderblue;}
        .htmlcode>h1   {color: blue;}
        .htmlcode>p {color: red;}
    </style>
</head>
<body>
<div class="htmlcode">
<h1>Esto es un encabezado</h1>
<p>Esto es un párrafo.</p>
</div>
</body>
</html>
"""

In [ ]:
# HTML
w = widgets.HTML(value=html_code)
display.display(w)

# Gradio

[Gradio](https://www.gradio.app/) es una biblioteca que permite crear interfaces de usuario. Estas interfaces se pueden utilizar para probar modelos de aprendizaje automático. 


```python

import gradio as gr

def greet(name):
    return "Hello " + name + "!"

iface = gr.Interface(fn=greet, inputs="text", outputs="text")
iface.launch(share=True)
```

```python
import gradio as gr

def calculate(num1, num2, operation):
	match operation:
		case "Suma":
			return num1 + num2
		case "Resta":
			return num1 - num2
		case "Multiplicación":
			return num1 * num2
		case "División":
			return num1 / num2 if num2 != 0 else "Error: División por cero"

with gr.Blocks() as app:
	with gr.Row():
		num1 = gr.Number(label="Número 1")
		num2 = gr.Number(label="Número 2")
	operation = gr.Radio(["Suma", "Resta", "Multiplicación", "División"], label="Operación")
	result = gr.Textbox(label="Resultado")
	calculate_button = gr.Button("Calcular")
	calculate_button.click(calculate, inputs=[num1, num2, operation], outputs=result)

app.launch()

```

## Stable Diffusion

### Instalar librerías

```python
# %pip install -U  torch torchvision torchaudio # Esto rompe kaggle con la p100 al menos
%pip install git+https://github.com/huggingface/diffusers gradio
%pip install --upgrade transformers
%pip install -Uqq peft accelerate transformers datasets bitsandbytes

```

### Descargar el modelo

```python

from diffusers import StableDiffusionPipeline
import torch

model_id = "sd-legacy/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to("cuda")

def infer(prompt, negative_prompt=None, width=512, height=512, num_inference_steps=50, guidance_scale=7.5
         ):
    image = pipe(prompt, negative_prompt=negative_prompt, num_inference_steps=num_inference_steps,
                 guidance_scale=guidance_scale, width=width, height=height).images[0]
    return image


```

### Generar Imagen

 ```python

import gradio as gr
with gr.Blocks() as app:
    with gr.Row():
        prompt = gr.Textbox(label="Prompt", placeholder="Enter your prompt here",value="a  close up photo of  son goku    in the forest")
    with gr.Row():
        negativeprompt = gr.Textbox(label="Negative Prompt", placeholder="Enter negative prompt here",value=" deformed iris, deformed pupils , text, cropped, out of frame, worst quality, low quality, jpeg artifacts, ugly, duplicate, morbid, mutilated, extra fingers, mutated hands, poorly drawn hands, poorly drawn face, mutation, deformed, blurry, dehydrated, bad anatomy, bad proportions, extra limbs, cloned face, disfigured, gross proportions, malformed limbs, missing arms, missing legs, extra arms, extra legs, fused fingers, too many fingers")
    with gr.Row():
        height = gr.Slider(label="Height", minimum=256, maximum=1024, step=64, value=512)
        width = gr.Slider(label="Width", minimum=256, maximum=1024, step=64, value=512)
    with gr.Row():
        steps = gr.Slider(label="Steps", minimum=5, maximum=200, step=1, value=30)
        scale = gr.Slider(label="Guidance Scale", minimum=1.0, maximum=15.0, step=0.1, value=7.5)
    generate_button = gr.Button("Generate Image")
    output = gr.Image(label="Generated Image")

    generate_button.click(
        infer,
        inputs=[prompt, negativeprompt, height, width, steps, scale],
        outputs=output
    )

app.launch(share=True)
 ```

# SANA by NVlabs (nvidia)


## Descargar y configurar modelo
```python
import torch
from diffusers import SanaPAGPipeline

pipe = SanaPAGPipeline.from_pretrained(
  "Efficient-Large-Model/Sana_1600M_1024px_diffusers",
  variant="fp16",
  torch_dtype=torch.float16,
  pag_applied_layers="transformer_blocks.8",
)
pipe.to("cuda")

pipe.text_encoder.to(torch.bfloat16)
pipe.vae.to(torch.bfloat16)
```

# Inferencia
```python
import gradio as gr

def infer(prompt="a close up photo of son goku in the forest", steps=20, scale=7.5):
    image = pipe(
        prompt=prompt,
        guidance_scale=scale,
        pag_scale=2.0,
        num_inference_steps=steps,
    )[0][0]
    return image


with gr.Blocks() as app:
    with gr.Row():
        prompt = gr.Textbox(
            label="Prompt",
            placeholder="Enter your prompt here",
            value="a  close up photo of  son goku    in the forest",
        )

    with gr.Row():
        steps = gr.Slider(label="Steps", minimum=5, maximum=200, step=1, value=20)
        scale = gr.Slider(
            label="Guidance Scale", minimum=1.0, maximum=15.0, step=0.1, value=7.5
        )
    generate_button = gr.Button("Generate Image")
    output = gr.Image(label="Generated Image")

    generate_button.click(
        infer,
        inputs=[prompt,  steps, scale],
        outputs=output,
    )

app.launch(share=True)
```

## Actividades

1. Agentes Inteligentes

Utilizando una celda Markdown, explique el concepto de **agente inteligente**.
Incluya una definición clara, características principales y ejemplos de aplicación en la vida real o en sistemas de software.

2. Sensores, Actuadores y Entornos

Utilizando una celda Markdown, explique qué son los **sensores**, los **actuadores** y los **entornos** en los que operan los agentes.
Incluya ejemplos concretos para cada uno, como sensores de temperatura, actuadores de movimiento, y entornos físicos o virtuales.

3. Comparación: Agentes Inteligentes vs Objetos en la Programación Orientada a Objetos

Utilizando una celda Markdown, realice una comparación entre los **agentes inteligentes** y los **objetos** en el paradigma de **programación orientada a objetos**.
Incluya similitudes y diferencias, enfocándose en la autonomía, el control del comportamiento, y la interacción con el entorno.

4. Clase Agente en Python

Utilizando una celda de código en Python, desarrolle una clase llamada `Agente`, la cual represente un agente simple.
Incluya atributos y métodos básicos como `percebir()`, `actuar()` y un na función que actualice su estado interno.

5. Aplicación en Gradio: Cálculos con un Número Entero (2 puntos)

Agregue una celda de código en Python que, al ejecutarse, muestre una **aplicación en Gradio**.
La interfaz debe recibir un valor entero y permitir seleccionar una de las siguientes opciones desde una lista:

- Calcular el **factorial** de un número.
- Mostrar la **tabla de multiplicar completa** del número.
- Mostrar la **tabla de multiplicar solo con resultados pares**.
- Mostrar la **tabla de multiplicar solo con resultados impares**.

6. Aplicación en Gradio: Operaciones entre Dos Números (2 puntos)

Agregue una celda de código en Python que defina las siguientes funciones y despliegue una **aplicación en Gradio** para operar sobre dos valores ingresados:

- `suma(a, b)`: retorna la suma de `a` y `b`.
- `resta(a, b)`: retorna la resta de `a` menos `b`.
- `multiplicacion(a, b)`: retorna la multiplicación de `a` y `b`.
- `division(a, b)`: retorna la división de `a` entre `b`.
- `potencia(a, b)`: retorna `a` elevado a la potencia `b`.

La aplicación debe permitir al usuario ingresar dos números y seleccionar la operación deseada desde una lista.

7. Lista de Personas (2 puntos)

**Paso 1:**
Agregue una celda de código en Python que, al ejecutarse, cree un archivo de texto en formato **CSV** con los encabezados: `nombre`, `apellido` y `edad`, y lo guarde en el disco.

**Paso 2:**
Agregue otra celda de código en Python que muestre una **aplicación en Gradio**.
Esta aplicación debe permitir al usuario ingresar el nombre, apellido y edad de una persona, y agregar esa información al archivo CSV creado anteriormente.

**Paso 3:**
Agregue una tercera celda de código en Python que, al ejecutarse, lea y muestre en pantalla el contenido completo del archivo CSV.
